复数morlet小波：
$$
\psi\left(t\right)=C_{\omega_0}\pi^{-\frac{1}{4}}e^{-\frac{1}{2}t^2}\left(e^{i\omega_0 t}-e^{-\frac{1}{2}\omega_0^2}\right)
$$
其中，$\omega_0$一般取6，$C_{\omega_0}=\left(1 + e^{-\omega_0^2} - 2 e^{-\frac{3}{4}\omega_0^2}\right)^{-\frac{1}{2}}$
此morlet小波与自身的对偶小波相同。因此：
小波变换：
$$
W\left(a,b\right)=\frac{1}{\sqrt{\left|a\right|}}\int_{-\infty }^{\infty}x\left(t\right)\bar{\psi}\left(\frac{t-b}{a}\right)\mathrm{d}t
$$
使用傅里叶变换计算：
$$
W\left(a,b\right)=\frac{\sqrt{a} }{2\pi}\int_{-\infty}^{\infty} e^{i\omega b}\hat{x}\left(\omega\right)\hat{\psi}\left(a\omega\right)\mathrm{d}\omega=\sqrt{a}  F^{-1}\left(\hat{x}\left(\omega\right)\hat{\psi}\left(a\omega\right)\right)
$$
写成求和形式：
$$
W(a_m, b_n) \approx \frac{1}{\sqrt{|a_m|}} \sum_{k} x(t_k) \bar{\psi}\left(\frac{t_k - b_n}{a_m}\right) \Delta t
$$
逆变换：
$$
x(t) = C_{\psi}^{-1} \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} W(a, b) \frac{1}{|a|^{1/2}} \tilde{\psi} \left( \frac{t-b}{a} \right) \, db \, \frac{da}{a^2}
$$
使用傅里叶变换进行计算：
$$
x\left ( t \right ) =-\frac{1}{C_{\psi}}\int_{-\infty}^{\infty} F^{-1}\left ( \hat{W}\left ( a,\Omega \right ) \hat{\psi}\left ( a\Omega \right )  \right )\frac{1}{a\sqrt{a}}\mathrm{d}a
$$
求和形式：
$$
x(t) = C_{\psi}^{-1} \sum_{i=0}^{n} \sum_{j=0}^{m} W(a_i, b_j) \frac{1}{|a_i|^{1/2}} \tilde{\psi} \left( \frac{t-b_j}{a_i} \right) \Delta b \frac{\Delta a}{a_i^2}
$$
其中，$C_{\psi}$是常数，定义为：
$$
C_{\psi} = \int_{-\infty}^{\infty} \frac{\bar{\hat{\psi}}(\omega) \hat{\tilde{\psi}}(\omega)}{|\omega|} d\omega
$$
$\hat{\psi}$表示$\psi$的傅里叶变换。$\omega_0=6$时，$C_{\psi}\approx 1$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
%matplotlib notebook

In [ ]:
def morlet_1D(t, omega0=6.0):
    C_omega0 = 1.0 / np.sqrt(1 + np.exp(-omega0**2) - 2*np.exp(-0.75*omega0**2))
    norm = np.pi**(-0.25)
    gauss = np.exp(-0.5 * t**2)
    wave_real = np.cos(omega0 * t) - np.exp(-0.5 * omega0**2)
    wave_imag = np.sin(omega0 * t)
    psi = (wave_real + 1j * wave_imag) * gauss

    return C_omega0 * norm * psi
def obtain_C_phi():
    C_phi = 0
    omega_arr_temp = np.linspace(-1000,1000,10000)
    d_omega = omega_arr_temp[1]-omega_arr_temp[0]
    for omega in omega_arr_temp:
        C_phi = C_phi + ft_morlet_1D(omega)*np.conj(ft_morlet_1D(omega))/np.abs(omega) * d_omega
    return C_phi
def ft_morlet_1D(omega, omega0=6.0):
    C_omega0 = 1.0 / np.sqrt(1 + np.exp(-omega0**2) - 2*np.exp(-0.75*omega0**2))
    return C_omega0 * np.pi**(-1.0/4) * (np.exp(-0.5*(omega-omega0)**2) - np.exp(-0.5*(omega**2+omega0**2)))

def cwt_morlet_1D(data, scales, omega0=6.0):
    N = len(data)
    S = len(scales)
    mean = np.mean(data)
    zero_mean_data = np.copy(data) - mean
    ft_coefs = np.zeros((S, N), dtype=np.complex128)
    omega = 2.0 * np.pi * np.fft.fftfreq(N)
    for i, a in enumerate(scales):
        ft_coefs[i, :] = np.sqrt(a) * np.fft.fft(zero_mean_data) * np.conj(ft_morlet_1D(a*omega, omega0=omega0))

    return np.fft.ifft(ft_coefs), mean

def icwt_morlet_1D(coe, scales, omega0=6.0):
    data_len = coe.shape[-1]
    omega = 2.0 * np.pi * np.fft.fftfreq(data_len)
    ft_x = np.zeros(data_len, dtype=np.complex128)
    da = scales[1] - scales[0]

    for i, scale in enumerate(scales):
        ft_morlet = ft_morlet_1D(scale*omega, omega0=omega0)
        ft_coe = np.fft.fft(coe[i, :])
        ft_x += scale**(-1.5) * da * ft_coe * ft_morlet
    C = obtain_C_phi()
    return np.fft.ifft(ft_x/C).real*2

In [ ]:
x = 5*np.sin(np.linspace(-2*np.pi, 2*np.pi, 100)) + np.random.rand(100)

In [ ]:
scales = np.linspace(0.1, 100, 1000)
coe, mean = cwt_morlet_1D(x, scales)

In [ ]:
x1 = icwt_morlet_1D(coe, scales, omega0=6.0) + mean

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.linspace(-2*np.pi, 2*np.pi, 100), x, label='signal')
ax.plot(np.linspace(-2*np.pi, 2*np.pi, 100), x1, label="ICWT")
ax.plot(np.linspace(-2*np.pi, 2*np.pi, 100), x1-x, label='difference')
ax.legend()

# 对单个信号进行滤波测试

In [ ]:
def gaussian(dx, mean, std):
    dx = np.array(dx)
    return mean * np.exp(-dx ** 2 / (2 * std ** 2))


def generate_signal(amplitude, omega, std_signal, mean_signal, std_noise, shape=(51, 51), seed=None):
    if seed is not None:
        np.random.seed(seed)
    noise = np.random.normal(mean_signal / 3, std_noise, shape)
    len_t, len_x = shape
    t_vect = np.linspace(0, len_t - 1, len_t).astype(int)
    x_vect = np.floor(
        len_x // 2 + amplitude * np.sin(omega * t_vect + np.random.uniform(0, 2 * np.pi)) + np.random.normal(0,
                                                                                                             amplitude / 4,
                                                                                                             len_t)).astype(
        int)
    signal = mean_signal * np.exp(-(np.arange(0, len_x)[None, :] - x_vect[:, None]) ** 2 / (2 * std_signal ** 2))
    signal += noise
    return signal

In [ ]:
signal = generate_signal(5, 0.08 * np.pi, 1, 10, 1, seed=100)
fig, ax = plt.subplots()
ax.imshow(signal, cmap='gray', origin='lower')
ax.set_xlabel('x')
ax.set_ylabel('t')
ax.set_title('signal with period of 25')
fig.savefig('./fig/amplitude/signal.png', dpi=300)

In [ ]:
phase = []
amplitude = []
coes = []
means = []
scales = np.arange(1, 100)
for i in range(signal.shape[-1]):
    coe, mean = cwt_morlet_1D(signal[:, i], scales=scales)
    phase.append(np.angle(coe))
    amplitude.append(np.abs(coe))
    coes.append(coe)
    means.append(mean)

amplitude = np.array(amplitude)
phase = np.array(phase)
coes = np.array(coes)
means = np.array(means)

In [ ]:
import matplotlib.animation as animation

def imshow_vedio(fig, data, cmap, **kwargs):
    data = np.array(data)
    ax = fig.axes[0]
    im = ax.imshow(data[0], cmap=cmap, origin='lower', vmin=np.min(data), vmax=np.max(data))
    ax.set_xlabel(kwargs.get('xlabel', 'x'))
    ax.set_ylabel(kwargs.get('ylabel', 'y'))
    ax.set_aspect('auto')
    title = ax.set_title(kwargs.get('title', 't=0'))
    plt.colorbar(im, ax=ax, label=kwargs.get('name_cbar', None))

    def update(frame):
        polt_data = data[:, frame, :].T
        im.set_data(polt_data)
        title.set_text(kwargs.get('title', f't')+f'={frame}')
        return [im, title]

    ani = animation.FuncAnimation(fig=fig, func=update, frames=data.shape[1], interval=kwargs.get('interval', 100))
    return ani

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ani = imshow_vedio(fig, amplitude, cmap='jet', xlabel='x', ylabel='t', title='scale', name_cbar='amplititude', interval=500)

In [ ]:
filter_coe = np.copy(coes)
filter_coe[np.abs(coes) < 1] = 0

In [ ]:
i_signal = []
for i in range(filter_coe.shape[0]):
    s = icwt_morlet_1D(filter_coe[i, :, :], scales=scales)
    i_signal.append(s+means[i])

i_signal = np.array(i_signal).T
fig, axs = plt.subplots(ncols=2, figsize=(9, 4))

im1 = axs[0].imshow(signal, cmap='gray', origin='lower')
axs[0].set_xlabel('x')
axs[0].set_ylabel('t')
axs[0].set_title('signal')
plt.colorbar(im1, ax=axs[0])
im2 = axs[1].imshow(i_signal, cmap='gray', origin='lower')
axs[1].set_xlabel('x')
axs[1].set_ylabel('t')
axs[1].set_title('icwt_signal')
plt.colorbar(im2, ax=axs[1])

# 对模拟时空图进行滤波测试

In [46]:
def cwt_filter(data, scales, threshold):
    coes = []
    means = []

    for i in range(data.shape[-1]):
        coe, mean = cwt_morlet_1D(data[:, i], scales=scales)
        coes.append(coe)
        means.append(mean)

    coes = np.array(coes)
    means = np.array(means)

    filtered_coes = np.copy(coes)
    threshold_data = np.percentile(np.abs(coes).flatten(), 100*(1-threshold))
    filtered_coes[np.abs(filtered_coes) < threshold_data] = 0

    i_data = []

    for i in range(filtered_coes.shape[0]):
        s = icwt_morlet_1D(filtered_coes[i, :, :], scales=scales)
        i_data.append(s+means[i])

    return np.array(i_data).T

In [ ]:
from scipy.stats import truncnorm

def simulation(x, t, dx=120, dt=3, num=200, seed=None):
    if seed is not None:
        np.random.seed(seed)

    life_span = {'mean': 100, 'std': 50} # s
    brightness = {'mean': 24, 'std': 8}
    period = {'min': 60, 'max': 120} # s
    amplitude = {'mean': 15, 'std': 3} # km/s
    width = 7
    noise = {'mean': 7, 'std': 17}

    x_vect = np.arange(0, x, dx)
    t_vect = np.arange(0, t, dt)

    space_time = np.zeros((len(t_vect), len(x_vect)))

    for i in range(num):
        begin_time = np.random.uniform(0, t)
        begin_x = np.random.uniform(0, x)
        life = truncnorm.rvs(-2, 3, loc=life_span['mean'], scale=life_span['std'])
        phase = np.random.uniform(0, 2*np.pi)
        T = np.random.uniform(period['min'], period['max'])
        I = truncnorm.rvs(-4, 4, loc=brightness['mean'], scale=brightness['std'])
        speed = truncnorm.rvs(-3, 3, loc=amplitude['mean'], scale=amplitude['std'])
        A = 0.5*np.cos(np.random.uniform(0, 0.5*np.pi))*speed*T/np.pi

        t_index = (np.arange(begin_time, begin_time+life, dt)/dt).astype(int)
        t_index = np.clip(t_index, 0, len(t_vect)-1)
        c_index = np.floor((begin_x + A*np.sin(2*np.pi * t_index*dt/T + phase) + np.random.normal(0, A/5, len(t_index)))/dx).astype(int)
        c_index = np.clip(c_index, 0, len(x_vect)-1)

        for j, time in enumerate(t_index):
            c = c_index[j]
            x_index = (np.array(range(width)) - width//2 + c).astype(int)
            x_index = np.clip(x_index, 0, len(x_vect)-1)
            for index in x_index:
                # space_time[time, index] = I * np.exp(-(index-c)**2 / (2 * (width/10)**2))
                space_time[time, index] = I * np.cos(np.pi/(2*(width//2))*(index-c))

    space_time += truncnorm.rvs(-5, 5, loc=noise['mean'], scale=noise['std'], size=(len(t_vect), len(x_vect)))
    return space_time

In [ ]:
st_data = simulation(301*120, 603, num=100, seed=0)
plt.imshow(st_data, cmap='gray', origin='lower')

In [48]:
plt.imshow(cwt_filter(data=st_data, scales=np.arange(1, 100), threshold=0.5), cmap='gray', origin='lower')

<IPython.core.display.Javascript object>